In [ ]:
import os
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm  # Progress bar for file processing

# Define dataset directory
dataset_dir = '/home/omj/Desktop/driver_activity_detection/state-farm-distracted-driver-detection/imgs/train'

# Check if dataset directory exists
if not os.path.exists(dataset_dir):
    raise FileNotFoundError(f"Directory does not exist: {dataset_dir}")
else:
    print(f"Directory found: {dataset_dir}")

# Output directory for split data
output_dir = './driver_split_dataset'

# Classes
classes = ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']

# Create output directories for train and test splits
train_dir = os.path.join(output_dir, 'train')
test_dir = os.path.join(output_dir, 'test')

for cls in classes:
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls), exist_ok=True)

# Image preprocessing parameters
img_size = (224, 224)  # ResNet expects 224x224 inputs

# Split each class into train and test sets
for cls in classes:
    cls_path = os.path.join(dataset_dir, cls)
    images = os.listdir(cls_path)
    train_images, test_images = train_test_split(images, test_size=0.2, random_state=42)

    # Process train images
    for img_name in tqdm(train_images, desc=f'Processing training images for {cls}'):
        img_path = os.path.join(cls_path, img_name)
        img = Image.open(img_path).convert('L')  # Convert to grayscale
        img = img.resize(img_size)  # Resize
        img.save(os.path.join(train_dir, cls, img_name))  # Save to train folder

    # Process test images
    for img_name in tqdm(test_images, desc=f'Processing testing images for {cls}'):
        img_path = os.path.join(cls_path, img_name)
        img = Image.open(img_path).convert('L')  # Convert to grayscale
        img = img.resize(img_size)  # Resize
        img.save(os.path.join(test_dir, cls, img_name))  # Save to test folder

print("Dataset split and preprocessing complete.")


: 

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm  # Progress bar for training

# Data directory
output_dir = './driver_split_dataset'

# Data transforms for training and testing
data_transforms = {
    'train': transforms.Compose([
        transforms.Grayscale(num_output_channels=3),  # Convert grayscale to 3-channel for ResNet
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Standard normalization
    ]),
    'test': transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Load datasets
train_dataset = datasets.ImageFolder(os.path.join(output_dir, 'train'), transform=data_transforms['train'])
test_dataset = datasets.ImageFolder(os.path.join(output_dir, 'test'), transform=data_transforms['test'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Use pre-trained ResNet50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(pretrained=True)

# Modify the final layer for 10 classes (driver activities)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)

model = model.to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Training the model
num_epochs = 20

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 10)
    
    # Training phase
    model.train()
    running_loss = 0.0
    running_corrects = 0

    # Use tqdm for the training progress bar
    for inputs, labels in tqdm(train_loader, desc=f'Training Epoch {epoch+1}/{num_epochs}'):
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.double() / len(train_dataset)

    print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

# Save the trained model
torch.save(model.state_dict(), './resnet50_driver_activity.pth')

print("Model training complete and saved as 'resnet50_driver_activity.pth'")

# Evaluation phase
model.eval()
running_corrects = 0

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc='Evaluating on Test Set'):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        running_corrects += torch.sum(preds == labels.data)

test_acc = running_corrects.double() / len(test_dataset)
print(f'Test Accuracy: {test_acc:.4f}')


In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm

# Paths
test_mixed_dir = '/home/omj/Desktop/driver_activity_detection/state-farm-distracted-driver-detection/imgs/test'
output_dir = '/home/omj/Desktop/driver_activity_detection/state-farm-distracted-driver-detection/imgs/test_dataset_separated'
model_path = './resnet50_driver_activity.pth'  # Path to the saved model

# Classes corresponding to predictions
classes = ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']

# Create output directories for each class
for cls in classes:
    os.makedirs(os.path.join(output_dir, cls), exist_ok=True)

# Image transformation: Grayscale, Resize to 224x224, ToTensor, and Normalize
img_size = (224, 224)
data_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels for ResNet
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Standard normalization
])

# Load the trained ResNet-50 model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(pretrained=False)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(classes))  # Modify for 10 output classes

# Load saved weights
model.load_state_dict(torch.load(model_path))
model = model.to(device)
model.eval()  # Set model to evaluation mode

# Process each image in the mixed test directory
for img_name in tqdm(os.listdir(test_mixed_dir), desc="Classifying test images"):
    img_path = os.path.join(test_mixed_dir, img_name)
    
    if img_name.endswith(('png', 'jpg', 'jpeg')):
        # Open the image, apply transformations
        img = Image.open(img_path).convert('L')  # Convert to grayscale
        input_tensor = data_transform(img).unsqueeze(0).to(device)  # Transform and add batch dimension
        
        # Predict the class
        with torch.no_grad():
            output = model(input_tensor)
            _, pred = torch.max(output, 1)  # Get the predicted class index

        predicted_class = classes[pred.item()]

        # Save the image in the predicted class folder in grayscale
        save_path = os.path.join(output_dir, predicted_class, img_name)
        img.save(save_path)  # Save as grayscale

print("Image classification and saving complete.")


In [ ]:
import os
import shutil
from tqdm import tqdm

# Paths
train_dir = '/home/omj/Desktop/driver_activity_detection/driver_split_dataset/train'
test_dir = '/home/omj/Desktop/driver_activity_detection/driver_split_dataset/test'
test_separated_dir = '/home/omj/Desktop/driver_activity_detection/state-farm-distracted-driver-detection/imgs/test_dataset_separated'
combined_dataset_dir = '/home/omj/Desktop/driver_activity_detection/state-farm-distracted-driver-detection/combined_dataset'

# Create the combined dataset directory
os.makedirs(combined_dataset_dir, exist_ok=True)

# Helper function to copy files
def copy_files(src_dir, dst_dir):
    """Copy all files from src_dir to dst_dir, preserving subdirectory structure."""
    for class_dir in os.listdir(src_dir):
        class_src_path = os.path.join(src_dir, class_dir)
        class_dst_path = os.path.join(dst_dir, class_dir)
        os.makedirs(class_dst_path, exist_ok=True)

        # Copy each image to the destination
        for img_name in os.listdir(class_src_path):
            src_img_path = os.path.join(class_src_path, img_name)
            dst_img_path = os.path.join(class_dst_path, img_name)
            shutil.copy(src_img_path, dst_img_path)

# Copy original training dataset to combined dataset folder
print("Copying original training dataset to combined dataset folder...")
copy_files(train_dir, combined_dataset_dir)

# Copy original testing dataset to combined dataset folder
print("Copying original testing dataset to combined dataset folder...")
copy_files(test_dir, combined_dataset_dir)

# Copy newly classified test dataset (test_dataset_separated) to combined dataset folder
print("Copying newly classified test dataset to combined dataset folder...")
copy_files(test_separated_dir, combined_dataset_dir)

print("Dataset merging complete!")


In [ ]:
import os
import shutil
import random
from tqdm import tqdm

# Paths
combined_dataset_dir = '/home/omj/Desktop/driver_activity_detection/state-farm-distracted-driver-detection/combined_dataset'
combined_split_data_dir = '/home/omj/Desktop/driver_activity_detection/state-farm-distracted-driver-detection/combined_split_data'

# Create directories for the split dataset
train_split_dir = os.path.join(combined_split_data_dir, 'train')
test_split_dir = os.path.join(combined_split_data_dir, 'test')
os.makedirs(train_split_dir, exist_ok=True)
os.makedirs(test_split_dir, exist_ok=True)

# Helper function to split files into train and test sets
def split_data(src_dir, train_dir, test_dir, split_ratio=0.8):
    """Split data into training and testing datasets."""
    for class_dir in os.listdir(src_dir):
        class_src_path = os.path.join(src_dir, class_dir)
        if os.path.isdir(class_src_path):  # Ensure it's a directory
            images = os.listdir(class_src_path)
            random.shuffle(images)  # Shuffle images
            
            # Calculate the split index
            split_index = int(len(images) * split_ratio)
            
            # Split into training and testing
            train_images = images[:split_index]
            test_images = images[split_index:]
            
            # Create directories for class in train and test
            class_train_path = os.path.join(train_dir, class_dir)
            class_test_path = os.path.join(test_dir, class_dir)
            os.makedirs(class_train_path, exist_ok=True)
            os.makedirs(class_test_path, exist_ok=True)
            
            # Copy images to respective directories
            for img_name in train_images:
                src_img_path = os.path.join(class_src_path, img_name)
                dst_img_path = os.path.join(class_train_path, img_name)
                shutil.copy(src_img_path, dst_img_path)

            for img_name in test_images:
                src_img_path = os.path.join(class_src_path, img_name)
                dst_img_path = os.path.join(class_test_path, img_name)
                shutil.copy(src_img_path, dst_img_path)

# Perform the split
print("Splitting combined dataset into train and test sets...")
split_data(combined_dataset_dir, train_split_dir, test_split_dir)

print("Dataset splitting complete!")


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
from tqdm import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
combined_split_data_dir = '/home/omj/Desktop/driver_activity_detection/state-farm-distracted-driver-detection/combined_split_data'
num_classes = 10  # Update this based on your number of classes
num_epochs = 20
batch_size = 32

# Data Augmentation and Normalization
transforms_train = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transforms_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Datasets
train_dataset = datasets.ImageFolder(os.path.join(combined_split_data_dir, 'train'), transform=transforms_train)
test_dataset = datasets.ImageFolder(os.path.join(combined_split_data_dir, 'test'), transform=transforms_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Load Pre-trained ResNet50 Model
model = models.resnet50(pretrained=True)

# Modify the final layer to match the number of classes
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# Adam Optimizer with Weight Decay
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Cosine Annealing Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

# Early Stopping Class
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

# Loss Function
criterion = nn.CrossEntropyLoss()

# Training Loop
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs):
    early_stopping = EarlyStopping(patience=7, min_delta=0)
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch"):
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        # Average loss
        running_loss /= len(train_loader)
        val_loss /= len(val_loader)
        print(f"Train Loss: {running_loss:.4f}, Validation Loss: {val_loss:.4f}")

        # Step the scheduler
        scheduler.step()

        # Check early stopping
        early_stopping(val_loss)
        if early_stopping.early_stop:
            print("Early stopping triggered")
            break

# K-Fold Cross Validation (optional)
# Uncomment if you want to implement K-Fold CV

# from sklearn.model_selection import KFold
# kfold = KFold(n_splits=5, shuffle=True, random_state=42)
# for fold, (train_ids, val_ids) in enumerate(kfold.split(train_dataset)):
#     train_sampler = torch.utils.data.SubsetRandomSampler(train_ids)
#     val_sampler = torch.utils.data.SubsetRandomSampler(val_ids)
#     train_loader = DataLoader(train_dataset, sampler=train_sampler, batch_size=batch_size)
#     val_loader = DataLoader(train_dataset, sampler=val_sampler, batch_size=batch_size)
#     train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs)

# Run the training
train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, num_epochs)

# Save the model
torch.save(model.state_dict(), "resnet50_driver_activity_detection.pth")
print("Model saved!")


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import datasets, models
from torchvision.models import ResNet50_Weights  # Import weights enum
from torch.utils.data import DataLoader
from tqdm import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
combined_split_data_dir = '/home/omj/Desktop/driver_activity_detection/state-farm-distracted-driver-detection/combined_split_data'
num_classes = 10  # Update this based on your number of classes
num_epochs = 20
batch_size = 32

# Data Augmentation and Normalization for Grayscale Images
transforms_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Convert to grayscale
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalization for single channel
])

transforms_test = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Convert to grayscale
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalization for single channel
])

# Datasets
train_dataset = datasets.ImageFolder(os.path.join(combined_split_data_dir, 'train'), transform=transforms_train)
test_dataset = datasets.ImageFolder(os.path.join(combined_split_data_dir, 'test'), transform=transforms_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Load Pre-trained ResNet50 Model with updated weights argument
weights = ResNet50_Weights.IMAGENET1K_V1  # Or use ResNet50_Weights.DEFAULT for the most up-to-date version
model = models.resnet50(weights=weights)

# Modify the first convolutional layer to handle single-channel (grayscale) images
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Modify the final layer to match the number of classes
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),  # Add dropout to prevent overfitting
    nn.Linear(model.fc.in_features, num_classes)
)

model = model.to(device)

# Adam Optimizer with Weight Decay
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Cosine Annealing Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

# Early Stopping Class
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

# Loss Function
criterion = nn.CrossEntropyLoss()

# Training Loop
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, accumulation_steps=2):
    early_stopping = EarlyStopping(patience=7, min_delta=0)
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for i, (images, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch")):
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            loss.backward()
            
            # Gradient accumulation
            if (i + 1) % accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()

            running_loss += loss.item()

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        # Average loss
        running_loss /= len(train_loader)
        val_loss /= len(val_loader)
        print(f"Train Loss: {running_loss:.4f}, Validation Loss: {val_loss:.4f}")

        # Step the scheduler
        scheduler.step()

        # Check early stopping
        early_stopping(val_loss)
        if early_stopping.early_stop:
            print("Early stopping triggered")
            break

# Run the training
train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, num_epochs)

# Save the model
torch.save(model.state_dict(), "resnet50_driver_activity_detection.pth")
print("Model saved!")


In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
from tqdm import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
combined_split_data_dir = '/home/omj/Desktop/driver_activity_detection/state-farm-distracted-driver-detection/combined_split_data'
num_classes = 10  # Update this based on your number of classes
batch_size = 32

# Test Dataset Transform
transforms_test = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Convert to grayscale
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalization for single channel
])

# Load the test dataset
test_dataset = datasets.ImageFolder(os.path.join(combined_split_data_dir, 'test'), transform=transforms_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Load Pre-trained ResNet50 Model
model = models.resnet50()

# Modify the first convolutional layer to handle single-channel (grayscale) images
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Modify the final layer to match the number of classes
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),  # Add dropout to prevent overfitting
    nn.Linear(model.fc.in_features, num_classes)
)

# Load the trained model's weights
model.load_state_dict(torch.load("resnet50_driver_activity_detection.pth"))

# Move the model to the device and set it to evaluation mode
model = model.to(device)
model.eval()

# Function to calculate accuracy
def test_model_accuracy(model, test_loader):
    correct = 0
    total = 0
    
    with torch.no_grad():  # No need to track gradients during testing
        for images, labels in tqdm(test_loader, desc="Evaluating", unit="batch"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy of the model on the test images: {accuracy:.2f}%')

# Test the model accuracy
test_model_accuracy(model, test_loader)


In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from tkinter import *
from tkinter import filedialog
from PIL import Image, ImageTk
import numpy as np

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the pre-trained ResNet50 model
model = models.resnet50()
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),  # Add dropout to prevent overfitting
    nn.Linear(model.fc.in_features, 10)
)

# Load the trained model's weights
model.load_state_dict(torch.load("resnet50_driver_activity_detection.pth"))
model = model.to(device)
model.eval()  # Set the model to evaluation mode

# Define the 10 classes
classes = [
    "c0: normal driving",
    "c1: texting - right",
    "c2: talking on the phone - right",
    "c3: texting - left",
    "c4: talking on the phone - left",
    "c5: operating the radio",
    "c6: drinking",
    "c7: reaching behind",
    "c8: hair and makeup",
    "c9: talking to passenger"
]

# Transform for input images
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Convert to grayscale
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Function to make predictions on a given image
def predict_image(image_path):
    img = Image.open(image_path).convert("RGB")  # Load and convert image to RGB
    img = transform(img).unsqueeze(0).to(device)  # Apply transformations and add batch dimension
    
    with torch.no_grad():
        outputs = model(img)  # Get model outputs
        probabilities = torch.softmax(outputs, dim=1).cpu().numpy()[0]  # Get probabilities

    return probabilities

# Function to display selected image and predictions
def load_image():
    file_path = filedialog.askopenfilename()
    if file_path:
        img = Image.open(file_path).convert("RGB")  # Open image in RGB mode
        img.thumbnail((1000, 1000))  # Resize for display in the GUI
        img_tk = ImageTk.PhotoImage(img)
        image_label.config(image=img_tk)
        image_label.image = img_tk  # Keep a reference to avoid garbage collection

        # Predict probabilities
        probabilities = predict_image(file_path)
        
        # Display probabilities
        for i, prob in enumerate(probabilities):
            prob_label[i].config(text=f"{classes[i]}: {prob * 100:.2f}%")

# Create the main Tkinter window
root = Tk()
root.title("Driver Activity Detection")

# Define a larger font for labels
label_font = ("Helvetica", 16)  # You can change the font family and size as needed

# Image display label
image_label = Label(root)
image_label.pack()

# Button to upload an image
upload_button = Button(root, text="Upload Image", command=load_image)
upload_button.pack()

# Labels to display the class probabilities with increased font size
prob_label = [Label(root, text="", font=label_font) for _ in range(10)]
for lbl in prob_label:
    lbl.pack()

# Start the Tkinter event loop
root.mainloop()


In [ ]:
# Install all necessary libraries
# ! pip install torch pillow numpy

import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from tkinter import *
from tkinter import filedialog
from tkinter import ttk
from PIL import Image, ImageTk
import numpy as np

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the pre-trained ResNet50 model
model = models.resnet50()
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),  # Add dropout to prevent overfitting
    nn.Linear(model.fc.in_features, 10)
)

# Load the trained model's weights with CPU compatibility
model.load_state_dict(torch.load("C:\\Users\\Acer\\Downloads\\resnet50_driver_activity_detection.pth", map_location=torch.device('cpu')))

model = model.to(device)
model.eval()  # Set the model to evaluation mode

# Define the 10 classes
classes = [
    "c0: normal driving",
    "c1: texting - right",
    "c2: talking on the phone - right",
    "c3: texting - left",
    "c4: talking on the phone - left",
    "c5: operating the radio",
    "c6: drinking",
    "c7: reaching behind",
    "c8: hair and makeup",
    "c9: talking to passenger"
]

# Transform for input images
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Convert to grayscale
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Function to make predictions on a given image
def predict_image(image_path):
    img = Image.open(image_path).convert("RGB")  # Load and convert image to RGB
    img = transform(img).unsqueeze(0).to(device)  # Apply transformations and add batch dimension
    
    with torch.no_grad():
        outputs = model(img)  # Get model outputs
        probabilities = torch.softmax(outputs, dim=1).cpu().numpy()[0]  # Get probabilities

    return probabilities

# Function to display selected image and predictions
def load_image():
    file_path = filedialog.askopenfilename()
    if file_path:
        img = Image.open(file_path).convert("RGB")  # Open image in RGB mode
        img.thumbnail((250, 250))  # Resize for display in the GUI
        img_tk = ImageTk.PhotoImage(img)
        image_label.config(image=img_tk)
        image_label.image = img_tk  # Keep a reference to avoid garbage collection

        # Predict probabilities
        probabilities = predict_image(file_path)

        # Get the class with the highest probability
        max_prob_class = np.argmax(probabilities)
        max_prob_value = np.max(probabilities)

        # Update label and color based on the class
        if max_prob_class == 0:
            status_label.config(text="SAFE", fg="white", bg="lightgreen")
        elif max_prob_class in [5, 6, 8, 9]:
            status_label.config(text="WARNING", fg="white", bg="orange")
        else:
            status_label.config(text="DANGER", fg="white", bg="red")

        # Update table with probabilities
        for i, prob in enumerate(probabilities):
            table_labels[i][1].config(text=f"{prob * 100:.2f}%")

            # Highlight the row with the maximum probability
            if prob == max_prob_value:
                table_labels[i][0].config(font=("Helvetica", 12, "bold"), relief="solid", borderwidth=3)
                table_labels[i][1].config(font=("Helvetica", 12, "bold"), relief="solid", borderwidth=3)
            else:
                table_labels[i][0].config(font=label_font, relief="solid", borderwidth=1)
                table_labels[i][1].config(font=label_font, relief="solid", borderwidth=1)

# Create the main Tkinter window
root = Tk()
root.title("Driver Activity Detection")
root.config(bg="white")  # Keep the main background white

# Define a larger font for labels
label_font = ("Helvetica", 12)  # Font for table content
header_font = ("Helvetica", 14, "bold")  # Font for table headers
status_font = ("Helvetica", 20, "bold")  # Font for status label in bold

# Center-align all widgets in root window
root.columnconfigure(0, weight=1)

# Create a label to show the safety status (Safe, Warning, Danger) at the top
status_label = Label(root, text="STATUS", font=status_font, width=20, height=2, bg="white", relief="solid", borderwidth=2)
status_label.grid(row=0, column=0, pady=10, padx=10)

# Image display label
image_label = Label(root, bg="white")
image_label.grid(row=1, column=0, pady=10)

# Pill-shaped Upload Button
upload_button = Button(root, text="Upload Image", command=load_image, font=label_font, bg="blue", fg="white", relief="raised", bd=2)
upload_button.config(width=10, height=2, borderwidth=2)
upload_button.grid(row=2, column=0, pady=10)

# Create a frame for the probability table
table_frame = Frame(root, bg="white")
table_frame.grid(row=3, column=0, pady=10)

# Table headers (matching the width of the rest of the columns)
class_header = Label(table_frame, text="Class", font=header_font, bg="white", relief="solid", width=30, borderwidth=1)
prob_header = Label(table_frame, text="Probability", font=header_font, bg="white", relief="solid", width=10, borderwidth=1)
class_header.grid(row=0, column=0, padx=5, pady=2)
prob_header.grid(row=0, column=1, padx=5, pady=2)

# Table to display class names and their probabilities
table_labels = []
for i in range(10):
    class_label = Label(table_frame, text=classes[i], font=label_font, bg="white", relief="solid", width=30, borderwidth=1)
    prob_label = Label(table_frame, text="0.00%", font=label_font, bg="white", relief="solid", width=7, borderwidth=1)
    class_label.grid(row=i+1, column=0, padx=5, pady=2)
    prob_label.grid(row=i+1, column=1, padx=5, pady=2)
    table_labels.append((class_label, prob_label))

# Center-align the table frame
table_frame.grid_columnconfigure(0, weight=1)
table_frame.grid_columnconfigure(1, weight=1)

# Center-align the entire content frame
root.grid_columnconfigure(0, weight=1)

# Start the Tkinter event loop
root.mainloop()

In [ ]:

import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from tkinter import *
from tkinter import filedialog
from tkinter import ttk
from PIL import Image, ImageTk
import numpy as np
import pygame  # Added pygame for sound


# Initialize pygame mixer
pygame.mixer.init()

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the pre-trained ResNet50 model
model = models.resnet50()
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),  # Add dropout to prevent overfitting
    nn.Linear(model.fc.in_features, 10)
)

# Load the trained model's weights
model.load_state_dict(torch.load("C:\\Users\\Acer\\Downloads\\resnet50_driver_activity_detection.pth", map_location=torch.device('cpu')))
model = model.to(device)
model.eval()  # Set the model to evaluation mode

# Define the 10 classes
classes = [
    "c0: normal driving",
    "c1: texting - right",
    "c2: talking on the phone - right",
    "c3: texting - left",
    "c4: talking on the phone - left",
    "c5: operating the radio",
    "c6: drinking",
    "c7: reaching behind",
    "c8: hair and makeup",
    "c9: talking to passenger"
]

# Transform for input images
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Convert to grayscale
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Function to stop any currently playing sound
def stop_sound():
    pygame.mixer.music.stop()

# Function to play sound on repeat
def play_sound(sound_file):
    # Stop any currently playing sound first
    stop_sound()
    
    # Load and play the sound file
    try:
        pygame.mixer.music.load(sound_file)
        pygame.mixer.music.play(-1)  # -1 means loop indefinitely
    except Exception as e:
        print(f"Error playing sound: {e}")

# Function to make predictions on a given image
def predict_image(image_path):
    img = Image.open(image_path).convert("RGB")  # Load and convert image to RGB
    img = transform(img).unsqueeze(0).to(device)  # Apply transformations and add batch dimension
    
    with torch.no_grad():
        outputs = model(img)  # Get model outputs
        probabilities = torch.softmax(outputs, dim=1).cpu().numpy()[0]  # Get probabilities

    return probabilities

# Function to display selected image and predictions
def load_image():
    # Stop any previously playing sound
    stop_sound()
    
    file_path = filedialog.askopenfilename()
    if file_path:
        img = Image.open(file_path).convert("RGB")  # Open image in RGB mode
        img.thumbnail((200, 200))  # Resize for display in the GUI
        img_tk = ImageTk.PhotoImage(img)
        image_label.config(image=img_tk)
        image_label.image = img_tk  # Keep a reference to avoid garbage collection

        # Predict probabilities
        probabilities = predict_image(file_path)

        # Get the class with the highest probability
        max_prob_class = np.argmax(probabilities)
        max_prob_value = np.max(probabilities)

        # Update label and color based on the class and play corresponding sound
        if max_prob_class == 0:
            status_label.config(text="SAFE", fg="white", bg="lightgreen")
            stop_sound()  # Ensure no sound plays for safe class
        elif max_prob_class in [5, 6, 8, 9]:
            status_label.config(text="WARNING", fg="white", bg="orange")
            play_sound("warning.mp3")  # Play warning sound
        else:
            status_label.config(text="DANGER", fg="white", bg="red")
            play_sound("danger.mp3")  # Play danger sound

        # Update table with probabilities
        for i, prob in enumerate(probabilities):
            table_labels[i][1].config(text=f"{prob * 100:.2f}%")

            # Highlight the row with the maximum probability
            if prob == max_prob_value:
                table_labels[i][0].config(font=("Helvetica", 12, "bold"), relief="solid", borderwidth=3)
                table_labels[i][1].config(font=("Helvetica", 12, "bold"), relief="solid", borderwidth=3)
            else:
                table_labels[i][0].config(font=label_font, relief="solid", borderwidth=1)
                table_labels[i][1].config(font=label_font, relief="solid", borderwidth=1)

# Create the main Tkinter window
root = Tk()
root.title("Driver Activity Detection")
root.config(bg="white")  # Keep the main background white

# Define a larger font for labels
label_font = ("Helvetica", 12)  # Font for table content
header_font = ("Helvetica", 14, "bold")  # Font for table headers
status_font = ("Helvetica", 20, "bold")  # Font for status label in bold

# Center-align all widgets in root window
root.columnconfigure(0, weight=1)

# Create a label to show the safety status (Safe, Warning, Danger) at the top
status_label = Label(root, text="STATUS", font=status_font, width=20, height=2, bg="white", relief="solid", borderwidth=2)
status_label.grid(row=0, column=0, pady=10, padx=10)

# Image display label
image_label = Label(root, bg="white")
image_label.grid(row=1, column=0, pady=10)

# Pill-shaped Upload Button
upload_button = Button(root, text="Upload Image", command=load_image, font=label_font, bg="blue", fg="white", relief="raised", bd=2)
upload_button.config(width=15, height=2, borderwidth=2)
upload_button.grid(row=2, column=0, pady=10)

# Create a frame for the probability table
table_frame = Frame(root, bg="white")
table_frame.grid(row=3, column=0, pady=10)

# Table headers (matching the width of the rest of the columns)
class_header = Label(table_frame, text="Class", font=header_font, bg="white", relief="solid", width=30, borderwidth=1)
prob_header = Label(table_frame, text="Probability", font=header_font, bg="white", relief="solid", width=10, borderwidth=1)
class_header.grid(row=0, column=0, padx=5, pady=2)
prob_header.grid(row=0, column=1, padx=5, pady=2)

# Table to display class names and their probabilities
table_labels = []
for i in range(10):
    class_label = Label(table_frame, text=classes[i], font=label_font, bg="white", relief="solid", width=30, borderwidth=1)
    prob_label = Label(table_frame, text="0.00%", font=label_font, bg="white", relief="solid", width=10, borderwidth=1)
    class_label.grid(row=i+1, column=0, padx=5, pady=2)
    prob_label.grid(row=i+1, column=1, padx=5, pady=2)
    table_labels.append((class_label, prob_label))

# Center-align the table frame
table_frame.grid_columnconfigure(0, weight=1)
table_frame.grid_columnconfigure(1, weight=1)

# Center-align the entire content frame
root.grid_columnconfigure(0, weight=1)

# Start the Tkinter event loop
root.mainloop()

# Quit pygame mixer when the window is closed
pygame.mixer.quit()